## 04. Syntax Analysis & NLP Pipeline

### 학습 목표
1. `detect_syntax`로 품사(POS) 태깅을 수행한다.
2. 앞서 배운 API를 통합하는 **종합 분석 파이프라인**을 구현한다.
3. 분석 결과를 JSON/CSV로 저장하고 시각화 대시보드를 만든다.

### API 개요
```python
# 구문 분석 (품사 태깅)
comprehend.detect_syntax(Text='...', LanguageCode='en')
```

### 주요 품사 태그
```
NOUN  VERB  ADJ  ADV  PROPN(고유명사)
DET   ADP   CONJ  PUNCT  NUM
```

### Syntax 결과 구조
```
{
  'TokenId': 1,
  'Text': 'The',
  'PartOfSpeech': {'Tag': 'DET', 'Score': 0.999},
  'BeginOffset': 0, 'EndOffset': 3
}
```


In [ ]:
#환경 초기화
import boto3, json, os
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

# 한글 폰트 설정 (SageMaker Studio)
try:
    import koreanize_matplotlib
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'koreanize-matplotlib'])
    import koreanize_matplotlib

comprehend = boto3.client('comprehend', region_name='us-east-1')
os.makedirs('./lab04_output', exist_ok=True)
print('환경 초기화 완료 (us-east-1)')


In [ ]:
#detect_syntax - 품사 태깅

syntax_text = (
    "Amazon Web Services provides reliable cloud computing solutions "
    "that help businesses scale quickly and efficiently."
)

response = comprehend.detect_syntax(
    Text=syntax_text,     
    LanguageCode='en' 
)

tokens = response['SyntaxTokens'] 
print(f'토큰 수: {len(tokens)}개')
print('-' * 50)
for tok in tokens:
    pos  = tok['PartOfSpeech']['Tag'] 
    conf = tok['PartOfSpeech']['Score'] 
    print(f"  {tok['Text']:<15} {pos:<8} ({conf:.3f})")


In [ ]:
# 품사 분포 시각화
pos_counts = Counter(t['PartOfSpeech']['Tag'] for t in tokens)
labels = list(pos_counts.keys())
sizes  = list(pos_counts.values())

plt.figure(figsize=(9, 4))
plt.bar(labels, sizes, color='#3498db')
plt.title('품사 분포 (Part-of-Speech)')
plt.ylabel('빈도')
plt.tight_layout()
plt.show()

nouns = [t['Text'] for t in tokens if t['PartOfSpeech']['Tag'] in ('NOUN','PROPN')]
print('주요 명사/고유명사:', nouns)


In [ ]:
# 종합 분석 파이프라인 구현 - analyze_text()

def analyze_text(text, lang='ko'):
    """Comprehend 종합 분석 파이프라인"""
    result = {'text': text, 'language': lang}

    # 1단계: 언어 자동 감지
    lang_resp = comprehend.detect_dominant_language(Text=text) 
    detected_lang = lang_resp['Languages'][0]['LanguageCode'] 
    result['detected_language'] = detected_lang

    # 2단계: 감성 분석
    sent_resp = comprehend.detect_sentiment(
        Text=text, LanguageCode=detected_lang 
    )
    result['sentiment']       = sent_resp['Sentiment']
    result['sentiment_scores']= sent_resp['SentimentScore'] 

    # 3단계: 개체 인식
    ent_resp = comprehend.detect_entities(
        Text=text, LanguageCode=detected_lang
    )
    result['entities'] = [
        {'text': e['Text'], 'type': e['Type'], 'score': round(e['Score'],3)}
        for e in ent_resp['Entities'] 
    ]

    # 4단계: 핵심 문구
    kp_resp = comprehend.detect_key_phrases(
        Text=text, LanguageCode=detected_lang 
    )
    result['key_phrases'] = [p['Text'] for p in kp_resp['KeyPhrases']]

    return result

# 테스트
test = '삼성전자가 2024년 서울에서 AI 반도체 신제품을 발표했습니다.'
out  = analyze_text(test)
print(f"언어: {out['detected_language']}")
print(f"감성: {out['sentiment']}")
print(f"개체: {out['entities']}")
print(f"핵심문구: {out['key_phrases']}")


In [ ]:
# 배치 파이프라인 & 결과 저장

headlines = [
    '현대차, 전기차 신모델 출시로 테슬라와 본격 경쟁 선언',
    '코스피 3,000선 돌파, 외국인 투자자 대규모 순매수',
    'BTS 월드투어 티켓 매진, 팬들 아쉬움 토로',
    '기상청, 이번 주말 전국 폭설 예보',
]

all_results = []
for i, h in enumerate(headlines):
    print(f'처리 중 [{i+1}/{len(headlines)}]: {h[:25]}...')
    r = analyze_text(h)
    all_results.append(r)

# JSON 저장
with open('./lab04_output/results.json', 'w') as f: 
    json.dump(all_results, f, ensure_ascii=False, indent=2)  
print('\n✅ results.json 저장 완료')

# CSV 저장
df = pd.DataFrame([{
    '헤드라인': r['text'][:30],
    '감성'    : r['sentiment'], 
    '개체수'  : len(r['entities']),
    '핵심문구': ', '.join(r['key_phrases'][:3])
} for r in all_results])
df.to_csv('./lab04_output/results.csv', index=False, encoding='utf-8-sig')
print('✅ results.csv 저장 완료')
print(df.to_string(index=False))


In [ ]:
# 종합 시각화 대시보드

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# [0,0] 감성 분포 파이차트
sent_counts = Counter(r['sentiment'] for r in all_results) 
colors_map  = {'POSITIVE':'#2ecc71','NEGATIVE':'#e74c3c','NEUTRAL':'#95a5a6','MIXED':'#f39c12'}
axes[0,0].pie(
    sent_counts.values(),
    labels=sent_counts.keys(),
    colors=[colors_map.get(k,'#95a5a6') for k in sent_counts.keys()],
    autopct='%1.0f%%'
)
axes[0,0].set_title('감성 분포')

# [0,1] 개체 유형 분포
all_ent_types = [e['type'] for r in all_results for e in r['entities']]
ent_cnt = Counter(all_ent_types)
axes[0,1].bar(ent_cnt.keys(), ent_cnt.values(), color='#3498db')
axes[0,1].set_title('개체 유형 분포')
axes[0,1].tick_params(axis='x', rotation=30)

# [1,0] 헤드라인별 개체 수
ent_counts = [len(r['entities']) for r in all_results]
short_labels = [r['text'][:12]+'...' for r in all_results]
axes[1,0].barh(short_labels, ent_counts, color='#9b59b6') 
axes[1,0].set_title('헤드라인별 개체 수')

# [1,1] 헤드라인별 핵심 문구 수
kp_counts = [len(r['key_phrases']) for r in all_results]
axes[1,1].bar(range(len(headlines)), kp_counts, color='#e67e22') 
axes[1,1].set_xticks(range(len(headlines)))
axes[1,1].set_xticklabels([f'뉴스{i+1}' for i in range(len(headlines))])
axes[1,1].set_title('헤드라인별 핵심 문구 수')

plt.suptitle('Amazon Comprehend 종합 분석 대시보드', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./lab04_output/dashboard.png', dpi=150)
plt.show()
print('✅ dashboard.png 저장 완료')
